In [33]:
import numpy as np
import pandas as pd
import io
import scipy as sp
from scipy.stats import qmc
from numpy.random import Generator, PCG64

# Individual TE data

In [3]:
folder = '../Data/'
# df_AM22 = pd.read_excel(folder+"Armstrong McKay 2022/science.abn7950_data_s1.xlsx",
#                         skiprows=[0,1,2,3,4],sheet_name='4) Estimates',
#                         skipfooter=40,index_col=1,header=[0,1])
df_TEs = pd.read_excel(folder+'TEsummary.xlsx',header=[0],index_col=0)

df_TEs.head()

,Tipping element,Collapse (years),Recovery (years),Threshold (°C) – Min.,Threshold (°C) – Est.,Threshold (°C) – Max.,Beta (1/°C),Max Impact (°C) – Global,Comments
TE,,,,,,,,,
REEF,Low-latitude coral reefs,10,100,1.0,1.5,2.0,13.813510,0.0,From Armstrong McKay et al no estimate for max...
ASI,Arctic winter sea ice,20,200,4.5,6.3,8.7,3.837086,0.6,NaN
PERM,Boreal permafrost collapse,50,500,3.0,4.0,6.0,6.906755,0.3,From Armstrong McKay 2022 max impact is estima...
AMOC,Atlantic meridional ocean circulation,50,500,1.4,4.0,8.0,2.656444,-0.5,NaN
WAM,Sahel and West African monsoon,50,500,2.0,2.8,3.5,8.633443,0.0,From Armstrong McKay et al no estimate for max...


In [5]:
## checking betas are correct
value_at_min = 0.1/100 # 0.1% at T_min
beta = np.log(999)/(df_TEs['Threshold (°C) – Est.'].to_numpy() - df_TEs['Threshold (°C) – Min.'].to_numpy())

In [16]:
TEs = df_TEs.index.to_list()
TE_to_ind = {te:i for i,te in enumerate(TEs)}
print(TE_to_ind)

{'REEF': 0, 'ASI': 1, 'PERM': 2, 'AMOC': 3, 'WAM': 4, 'AMAZ': 5, 'ENSO': 6, 'WAIS': 7, 'GIS': 8}


# Generating interaction matrices

In [8]:
folder = '../Data/'
df_interactions = pd.read_excel(folder+'Interactions_TableA2.xlsx',usecols='A:F',header=[0])
df_interactions.head()

,Source,Target,Response,Strength,min,max
0,GIS,AMOC,Destabilising,Strong,0.1,10.0
1,WAIS,AMOC,Unclear,"Weak, Moderate",-3.0,3.0
2,AMOC,GIS,Stabilising,Strong,-10.0,-0.1
3,AMOC,WAIS,Destabilising,Unclear,0.1,1.5
4,GIS,WAIS,Destabilising,Moderate,0.1,5.0


In [32]:
# Example adjacency matrix using only the max values
A = np.zeros([len(TEs)]*2)
for index, row in df_interactions.iterrows():
    print(index)
    i = TE_to_ind[row['Source']]
    j = TE_to_ind[row['Target']]
    A[i,j] = row['max']

print(A)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
[[ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   2.   3.   0.   0.   0.   0.   2. ]
 [ 0.   0.   0.   1.5  0.   0.   0.   0.   0. ]
 [ 0.  -0.1  0.   0.   1.5  5.   2.   1.5 -0.1]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [10.   0.   0.   0.   0.  10.   0.   3.   0. ]
 [ 0.   0.   0.   3.   0.   0.   0.   0.   2. ]
 [ 0.   0.   0.  10.   0.   0.   0.   5.   0. ]]


In [52]:
no_samples = 500
sampler = qmc.LatinHypercube(d=len(df_interactions),rng=np.random.default_rng(seed=1850201420152350))
sample = sampler.random(n=no_samples)

In [56]:
a = df_interactions['min'].to_numpy()
b = df_interactions['max'].to_numpy()
folder_out = "../Data/Interaction matrices/"
for r in range(no_samples):
    weights = a+sample[r,:]*(b-a)
    A = np.zeros([len(TEs)]*2)
    for index, row in df_interactions.iterrows():
        i = TE_to_ind[row['Source']]
        j = TE_to_ind[row['Target']]
        A[i,j] = weights[index]
    np.savetxt(folder_out+str(r)+'.csv',A)